# Allen ABC Atlas: Spatial Transcriptomics

Explore cell metadata, spatial coordinates, and marker-gene expression from the Allen Brain Cell (ABC) Atlas.

## Quick start

Run this notebook from top to bottom for the default offline result: a small simulated spatial cell dataset, plots, and marker summaries. The ABC metadata section is an advanced live-data preview, separate from the synthetic analysis; leave it disabled unless you have reviewed the Atlas documentation, selected a table, and are ready to cache it.

## Prerequisites

- Python 3.9+ and familiarity with pandas DataFrames.
- Basic single-cell terminology: cells, genes, counts, normalization, and cell types.
- Optional: several GB of disk space and internet access if you choose to fetch ABC Atlas data.

## Setup

Install only the packages needed for the default offline analysis. The official Allen access client is an optional dependency in the advanced live-data section below.

In [ ]:
# Default offline tutorial dependencies
!pip install -q pandas numpy matplotlib seaborn

## Advanced preview (optional): discover a release, then cache reviewed metadata

This advanced live-data preview is separate from the synthetic analysis below. The official `AbcProjectCache` manages the current manifest and exposes `cache.manifest_file_names`; this avoids embedding a release identifier or a direct object-store URL that can become stale. First opt in to manifest discovery, inspect the printed manifest paths, and set `ABC_MANIFEST` to one of those exact values if you need a non-default release. Then inspect a directory's metadata names and explicitly choose a **modest** table. Metadata tables are downloaded into the relative cache directory; expression `.h5ad` matrices are intentionally not loaded here because they can be large.

In [ ]:
# Optional live-data dependency; run only when enabling RUN_ABC_LIVE.
# !pip install -q abc-atlas-access


In [ ]:
from pathlib import Path

RUN_ABC_LIVE = False  # Set True only after reviewing the current ABC Atlas documentation and data terms.
ABC_MANIFEST = None  # Optional: paste one exact path printed by this cell after reviewing it.
ABC_DIRECTORY = None  # Optional: paste one reviewed directory name, e.g. from list_metadata_files(...).
ABC_METADATA_FILE = None  # Optional: paste one reviewed, modest metadata-table name.
CONFIRM_MODEST_METADATA_DOWNLOAD = False  # Required before any metadata table is downloaded.
abc_cache_dir = Path('data') / 'abc_atlas'
live_metadata = None

if RUN_ABC_LIVE:
    try:
        from abc_atlas_access.abc_atlas_cache.abc_project_cache import AbcProjectCache
        abc_cache = AbcProjectCache.from_cache_dir(abc_cache_dir)
        available_manifests = list(abc_cache.cache.manifest_file_names)
        print('Cache directory:', abc_cache_dir)
        print('Current manifest:', abc_cache.current_manifest)
        print('Available manifest paths:')
        print(*available_manifests, sep='\n')
        if ABC_MANIFEST is not None:
            if ABC_MANIFEST not in available_manifests:
                raise ValueError('ABC_MANIFEST must exactly match one printed available manifest path.')
            abc_cache.load_manifest(ABC_MANIFEST)
            print('Loaded reviewed manifest:', abc_cache.current_manifest)

        if ABC_DIRECTORY is not None:
            metadata_names = abc_cache.list_metadata_files(ABC_DIRECTORY)
            print(f'Metadata files in {ABC_DIRECTORY}:', metadata_names)
            if ABC_METADATA_FILE is not None:
                if ABC_METADATA_FILE not in metadata_names:
                    raise ValueError('ABC_METADATA_FILE must exactly match a listed metadata file.')
                if not CONFIRM_MODEST_METADATA_DOWNLOAD:
                    raise RuntimeError('Review the table size/use case, then set CONFIRM_MODEST_METADATA_DOWNLOAD = True.')
                live_metadata = abc_cache.get_metadata_dataframe(
                    directory=ABC_DIRECTORY, file_name=ABC_METADATA_FILE
                )
                print('Loaded reviewed metadata table:', live_metadata.shape)
                display(live_metadata.head())
        else:
            print('Set ABC_DIRECTORY only after selecting a dataset in the Atlas/documentation.')
    except Exception as error:
        print('Live ABC path was not completed; using illustrative fallback:', error)
else:
    print('Live access disabled. Continuing with the small illustrative dataset.')

## Build a small spatial cell table

The fallback models three spatially separated cell groups and two marker genes. This is useful for testing plotting and aggregation code but does not reproduce the biological complexity, sequencing depth, coordinate system, or taxonomy of ABC Atlas data. The live path above exposes the selected table for inspection; check its release-specific column names and coordinate semantics before adapting it to these plots.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

rng = np.random.default_rng(42)
groups = {'Excitatory': (0.25, 0.67), 'Inhibitory': (0.65, 0.56), 'Glial': (0.47, 0.25)}
parts = []
for cell_type, (cx, cy) in groups.items():
    n = 120
    x, y = rng.normal(cx, 0.075, n), rng.normal(cy, 0.065, n)
    # Counts are simulated and intentionally retained as counts before log1p.
    slc17a7_mean = 5.0 if cell_type == 'Excitatory' else 0.25
    gad1_mean = 4.5 if cell_type == 'Inhibitory' else 0.2
    parts.append(pd.DataFrame({'cell_type': cell_type, 'x': x, 'y': y,
        'Slc17a7': rng.poisson(slc17a7_mean, n), 'Gad1': rng.poisson(gad1_mean, n)}))
cells = pd.concat(parts, ignore_index=True)
cells['log1p_Slc17a7'] = np.log1p(cells['Slc17a7'])
cells['log1p_Gad1'] = np.log1p(cells['Gad1'])
cells.head()

## Plot cell locations and spatial gene expression

Each point is a cell. Coordinates are unitless in the fallback; do not label them as CCF coordinates. Log-transforming count-like measurements improves visual dynamic range, but normalizing real expression matrices should follow the release-specific preprocessing documentation.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4), constrained_layout=True)
sns.scatterplot(data=cells, x='x', y='y', hue='cell_type', s=22, linewidth=0, ax=axes[0])
axes[0].set(title='Illustrative spatial cell types', xlabel='x', ylabel='y')
for ax, gene in zip(axes[1:], ('log1p_Slc17a7', 'log1p_Gad1')):
    image = ax.scatter(cells['x'], cells['y'], c=cells[gene], cmap='magma', s=20, linewidths=0)
    ax.set(title=gene.replace('log1p_', '') + ' expression', xlabel='x', ylabel='y')
    fig.colorbar(image, ax=ax, label='log1p(count)')
plt.show()

## Summarize marker expression by cell type

Cell-level observations are not independent biological replicates. This group summary is descriptive; comparisons across animals, regions, or conditions should use the experimental unit and a model appropriate to the selected ABC dataset.

In [ ]:
summary = (cells.groupby('cell_type')[['Slc17a7', 'Gad1']]
           .agg(['mean', 'median', 'size'])
           .round(2))
display(summary)

ax = cells.melt(id_vars='cell_type', value_vars=['log1p_Slc17a7', 'log1p_Gad1'],
                var_name='gene', value_name='log_expression').pipe(
    sns.boxplot, x='cell_type', y='log_expression', hue='gene')
ax.set(title='Marker distributions by illustrative cell type', xlabel='cell type', ylabel='log1p(count)')
plt.xticks(rotation=15)
plt.show()

## References

- Allen Institute for Brain Science. Allen Brain Cell (ABC) Atlas. https://atlas.brain-map.org/
- Allen Institute. `abc_atlas_access` documentation. https://alleninstitute.github.io/abc_atlas_access/
- Yao, Z., van Velthoven, C. T. J., Nguyen, T. N., et al. (2021). A taxonomy of transcriptomic cell types across the isocortex and hippocampal formation. *Cell*, 184(12), 3222–3241.e26. https://doi.org/10.1016/j.cell.2021.04.021

## License

This notebook is licensed under the [Creative Commons Attribution 4.0 International License](https://creativecommons.org/licenses/by/4.0/). ABC Atlas data, software, and derived analyses are subject to their respective Allen Institute terms and licenses.